In [1]:
from rdkit import Chem
import pandas as pd

# Load the A1AUU analogs from the SDF file using RDKit
a1auu_supplier = Chem.SDMolSupplier("MU8843-Analogs.sdf")

# Initialize lists to hold the Compound IDs and SMILES
compound_ids = []
smiles = []

# Loop over each molecule in the SDF file
for i, mol in enumerate(a1auu_supplier):
    if mol is not None:  # Ensure the molecule is valid
        # Generate a unique Compound ID for each analog based on its index in the SDF file
        compound_ids.append(f"Analog_{i+1}")  # Analog_1, Analog_2, ...
        
        # Get the SMILES string from the molecule
        smiles.append(Chem.MolToSmiles(mol))  # Convert molecule to SMILES string

# Create a DataFrame from the lists (Compound ID and SMILES)
a1auu_df = pd.DataFrame({
    'Compound ID': compound_ids,
    'SMILES': smiles
})

# Now, you can proceed with cleaning the SMILES and the rest of your workflow
print(a1auu_df.head())  # To verify the first few rows of the DataFrame


  Compound ID                                             SMILES
0    Analog_1        CC(C)n1ncc2c(C(=O)Nc3nncs3)cc(-c3cccs3)nc21
1    Analog_2   CC(C)n1ncc2c(/C([O-])=N/c3nncs3)cc(-c3cccs3)nc21
2    Analog_3       Cc1nnc(NC(=O)c2cc(-c3cccs3)nc3c2cnn3C(C)C)s1
3    Analog_4  Cc1n[nH]/c(=N\C(=O)c2cc(-c3cccs3)nc3c2cnn3C(C)...
4    Analog_5  Cc1nnc(/N=C(\[O-])c2cc(-c3cccs3)nc3c2cnn3C(C)C)s1


In [2]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from openpyxl.styles import Alignment, Font
import io
import subprocess
import re
from rdkit import DataStructs
from rdkit import RDLogger

# Disable RDKit warnings
RDLogger.DisableLog('rdApp.warning')

# Function to clean SMILES using Open Babel
def clean_smiles_with_obabel(smiles):
    try:
        # Save SMILES to a temporary file
        with open("temp.smi", "w") as f:
            f.write(smiles)
        
        # Run the Open Babel command to clean the SMILES, redirect stdout and stderr to subprocess.PIPE
        result = subprocess.run(
            ['obabel', 'temp.smi', '-O', 'temp_output.smi'],
            stdout=subprocess.PIPE,  # Suppress standard output
            stderr=subprocess.PIPE,  # Suppress error output
            check=True
        )
        
        # Read the cleaned SMILES back from the output file
        with open("temp_output.smi", "r") as f:
            cleaned_smiles = f.readline().strip()
        
        return cleaned_smiles
    except Exception as e:
        print(f"Error cleaning SMILES: {e}")
        return None

# Load Cache Challenge Data
cache_challenge_df = pd.read_excel("all_data_with_structures_vs2.xlsx", usecols=['Smiles', 'KD (M)'])

# Load A1AUU analogs from the SDF file
a1auu_supplier = Chem.SDMolSupplier("MU8843-Analogs.sdf")

# Initialize lists to hold the Compound IDs and SMILES
compound_ids = []
smiles = []

# Loop over each molecule in the SDF file and extract SMILES and generate Compound ID
for i, mol in enumerate(a1auu_supplier):
    if mol is not None:  # Ensure the molecule is valid
        # Generate a unique Compound ID for each analog based on its index in the SDF file
        compound_ids.append(f"Analog_{i+1}")  # Analog_1, Analog_2, ...
        
        # Get the SMILES string from the molecule
        smiles.append(Chem.MolToSmiles(mol))  # Convert molecule to SMILES string

# Create a DataFrame from the lists (Compound ID and SMILES)
a1auu_df = pd.DataFrame({
    'Compound ID': compound_ids,
    'SMILES': smiles
})

# Clean SMILES strings using Open Babel function
cache_challenge_df['Smiles'] = cache_challenge_df['Smiles'].apply(clean_smiles_with_obabel)
a1auu_df['SMILES'] = a1auu_df['SMILES'].apply(clean_smiles_with_obabel)

# Calculate Morgan Fingerprints
def calculate_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)

cache_challenge_df['Fingerprint'] = cache_challenge_df['Smiles'].apply(calculate_fingerprint)
a1auu_df['Fingerprint'] = a1auu_df['SMILES'].apply(calculate_fingerprint)

# Function to calculate Tanimoto similarity
def tanimoto_similarity(fp1, fp2):
    return DataStructs.TanimotoSimilarity(fp1, fp2)

# Calculate Tanimoto similarity between A1AUU analogs and Cache Challenge compounds
similarities = []
for _, analog_row in a1auu_df.iterrows():
    analog_smiles = analog_row['SMILES']
    analog_fp = analog_row['Fingerprint']
    for _, challenge_row in cache_challenge_df.iterrows():
        challenge_fp = challenge_row['Fingerprint']
        similarity_score = tanimoto_similarity(analog_fp, challenge_fp)
        similarities.append((analog_row['Compound ID'], challenge_row['Smiles'], challenge_row['KD (M)'], similarity_score))

# Convert similarities to a DataFrame
similarities_df = pd.DataFrame(similarities, columns=['Analog Compound', 'Cache Challenge Compound', 'Cache KD (M)', 'Similarity'])

# Select top 5 most similar compounds per analog
def select_top5_per_analog(df, threshold=0.4):
    top5_list = []
    for analog, group in df.groupby('Analog Compound'):
        group_sorted = group.sort_values(by='Similarity', ascending=False)
        high_sim = group_sorted[group_sorted['Similarity'] >= threshold]
        top5 = high_sim.head(5) if len(high_sim) >= 5 else group_sorted.head(5)
        top5_list.append(top5)
    return pd.concat(top5_list, ignore_index=True)

top5_similar_per_analog = select_top5_per_analog(similarities_df)

# Handle missing KD (M) values
top5_similar_per_analog['Cache KD (M)'] = top5_similar_per_analog['Cache KD (M)'].fillna(1)

# Calculate SkNN scores
def calculate_skNN_score(df):
    scores = []
    for analog in df['Analog Compound'].unique():
        analog_data = df[df['Analog Compound'] == analog]
        analog_data = analog_data.sort_values(by='Similarity', ascending=False)
        skNN_score = 0
        for rank, (index, row) in enumerate(analog_data.iterrows(), start=1):
            similarity = row['Similarity']
            KD = row['Cache KD (M)']
            binary_label = 1 if KD < 1 else 0
            skNN_score += (similarity ** 2 * binary_label) / rank
        scores.append({'Analog Compound': analog, 'SkNN Score': skNN_score})
    return pd.DataFrame(scores)

skNN_scores_df = calculate_skNN_score(top5_similar_per_analog)
skNN_scores_df_sorted = skNN_scores_df.sort_values(by='SkNN Score', ascending=False)

# Merge the data to get SMILES and generate images
merged_df = pd.merge(skNN_scores_df_sorted, a1auu_df[['Compound ID', 'SMILES']], left_on='Analog Compound', right_on='Compound ID', how='left')

# Function to generate image from SMILES
def smiles_to_image(smiles):
    mol = Chem.MolFromSmiles(smiles)
    img = Draw.MolToImage(mol, size=(300, 300))
    return img

# Generate images for all analogs
image_paths = []
for idx, row in merged_df.iterrows():
    img = smiles_to_image(row['SMILES'])
    img_byte_arr = io.BytesIO()
    img.save(img_byte_arr, format='PNG')
    img_byte_arr.seek(0)
    image_paths.append(img_byte_arr)

# Create a new workbook for final output
wb = Workbook()
ws = wb.active

# Add headers and make them bold
header_row = ["Analog Compound", "SkNN Score", "Structure (SMILES)", "Analog Images"]
ws.append(header_row)

# Bold the header row and set font size
for cell in ws[1]:
    cell.font = Font(bold=True, size=14)

# Adjust column widths and wrap text for better visibility
ws.column_dimensions['A'].width = 20
ws.column_dimensions['B'].width = 15
ws.column_dimensions['C'].width = 40
ws.column_dimensions['D'].width = 20

# Center text alignment for all header columns
for col in ws.columns:
    for cell in col:
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# Write the data and embed images
for idx, row in merged_df.iterrows():
    ws.append([row['Analog Compound'], row['SkNN Score'], row['SMILES']])
    img = Image(image_paths[idx])
    img.width = 150
    img.height = 150
    cell = ws.cell(row=idx + 2, column=4)
    ws.add_image(img, cell.coordinate)
    ws.row_dimensions[idx + 2].height = 150

# Save the final Excel file
wb.save('A1AUU_Analog_SkNN_Scores_with_SMILES_and_Images_Formatted.xlsx')

print("Excel file saved with images and formatting successfully.")


Excel file saved with images and formatting successfully.


In [3]:
filtered_analog_df = skNN_scores_df_sorted[skNN_scores_df_sorted['SkNN Score'] > 0.1]


In [4]:
from rdkit.Chem import SDWriter

# Filter the analogs with SkNN score greater than 0.1
filtered_analog_df = skNN_scores_df_sorted[skNN_scores_df_sorted['SkNN Score'] > 0.1]

# Create an SDWriter object for the new SDF file
output_sdf = SDWriter('Filtered_A1AUU_Analogs.sdf')

# Loop through the filtered analogs and write them to the new SDF file
for _, row in filtered_analog_df.iterrows():
    analog_id = row['Analog Compound']
    
    # Get the corresponding molecule from the A1AUU analogs list
    analog_mol = a1auu_supplier[int(analog_id.split('_')[1]) - 1]  # Index is based on 'Analog_1', 'Analog_2', etc.
    
    # If the molecule is not None, write it to the new SDF file
    if analog_mol is not None:
        output_sdf.write(analog_mol)

# Close the SDWriter after writing all the molecules
output_sdf.close()

print("Filtered analogs with SKNN scores > 0.1 saved to 'Filtered_A1AUU_Analogs.sdf'.")


Filtered analogs with SKNN scores > 0.1 saved to 'Filtered_A1AUU_Analogs.sdf'.


In [6]:
import openpyxl
from openpyxl.styles import Alignment

# Load the Excel file for MU8843
wb = openpyxl.load_workbook('A1AUU_Analog_SkNN_Scores_with_SMILES_and_Images_Formatted.xlsx')
ws = wb.active

# Define columns for alignment and wrapping
columns_to_center = ['A', 'B', 'C']  # Columns "Analog Compound", "SkNN Score", "Structure (SMILES)"

# Center and wrap text for the specified columns
for col in columns_to_center:
    for cell in ws[col]:
        # Apply both horizontal and vertical centering
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# Adjust the "Structure (SMILES)" column for wrapping (specifically for SMILES)
ws.column_dimensions['C'].width = 40  # Adjust width for better visibility

# Save the modified workbook
wb.save('MU8843_Analog_SkNN_Scores_with_SMILES_and_Images_Formatted_Updated.xlsx')

print("The file with updated formatting has been saved as 'MU8843_Analog_SkNN_Scores_with_SMILES_and_Images_Formatted_Updated.xlsx'.")


The file with updated formatting has been saved as 'MU8843_Analog_SkNN_Scores_with_SMILES_and_Images_Formatted_Updated.xlsx'.
